# Final Results Consolidation for FREIGHT-MNet

This notebook consolidates the full FREIGHT-MNet experiment chain into paper-ready tables, plots, and a final interpretation report.

It does **not** train new models. It loads and standardizes outputs from:

- M0 / M0.5 / MLP temporal baselines
- Cold-OD non-graph baselines
- GraphSAGE / HGT Cold-OD baselines
- Graph ablations
- Topology ablations
- Disruption gate experiments
- Calibration and decision-aware CVaR-style evaluation

The notebook is defensive: missing result files are logged and skipped, sparse segment diagnostics are repaired through `row_id`, and reports are generated without `tabulate`.

## 1. Imports and environment notes

This consolidation notebook only needs tabular Python packages. It does not import PyTorch Geometric. If pandas cannot import in the active Jupyter kernel because of NumPy ABI problems, switch to a clean project environment before running.

In [1]:
from __future__ import annotations

import json
import os
import platform
import re
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib available:", MATPLOTLIB_AVAILABLE)

Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Python version: 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
Platform: Windows-10-10.0.26200-SP0
NumPy: 2.4.5
Pandas: 2.3.3
Matplotlib available: True


## 2. Configuration and project paths

The default paths follow the project structure used by all previous notebooks. Update `data_root` only if the project folder is moved.

In [2]:
@dataclass(frozen=True)
class ConsolidationConfig:
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")
    scope: str = "east_plus_gulf"
    run_name: str = "final_results_consolidation_v1_notebook"
    expected_cold_test_rows: int = 1057
    expected_temporal_test_rows: int = 10574
    overwrite: bool = True
    top_n_plot_rows: int = 20


@dataclass(frozen=True)
class ProjectPaths:
    cfg: ConsolidationConfig

    @property
    def output_dir(self) -> Path:
        return self.cfg.data_root / "10_experiments" / self.cfg.run_name / self.cfg.scope

    @property
    def tables_dir(self) -> Path:
        return self.output_dir / "tables"

    @property
    def plots_dir(self) -> Path:
        return self.output_dir / "plots"

    @property
    def reports_dir(self) -> Path:
        return self.output_dir / "reports"

    @property
    def supervised_model_ready_path(self) -> Path:
        return (
            self.cfg.data_root / "08_processed" / "model_ready"
            / f"freight_mnet_supervised_edges_2018_2024_{self.cfg.scope}.parquet"
        )

    @property
    def supervised_with_disruption_path(self) -> Path:
        return (
            self.cfg.data_root / "08_processed" / "disruption_features"
            / f"freight_mnet_supervised_edges_2018_2024_{self.cfg.scope}_with_disruption.parquet"
        )

    @property
    def manifest_path(self) -> Path:
        return (
            self.cfg.data_root / "08_processed" / "model_ready" / "_metadata"
            / "freight_mnet_supervised_feature_manifest.json"
        )

    @property
    def topology_features_path(self) -> Path:
        return (
            self.cfg.data_root / "08_processed" / "graph_inputs"
            / f"topology_features_od_{self.cfg.scope}.parquet"
        )

    @property
    def full_heterodata_path(self) -> Path:
        return (
            self.cfg.data_root / "08_processed" / "graph_inputs"
            / f"freight_mnet_full_heterodata_{self.cfg.scope}.pt"
        )


cfg = ConsolidationConfig()
paths = ProjectPaths(cfg)

for directory in [paths.output_dir, paths.tables_dir, paths.plots_dir, paths.reports_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("Output directory:", paths.output_dir)

Output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\final_results_consolidation_v1_notebook\east_plus_gulf


## 3. Utility functions

These functions safely read available files, normalize schemas, save outputs, and avoid optional Markdown dependencies.

In [3]:
def now_unix() -> float:
    return time.time()


def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return str(value)


def write_json(payload: Mapping[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, default=json_default)


def print_frame(title: str, frame: pd.DataFrame, max_rows: int = 12) -> None:
    print(f"\n{title}")
    if frame is None or frame.empty:
        print("(empty)")
    else:
        print(frame.head(max_rows).to_string(index=False))


def dataframe_to_text_table(frame: pd.DataFrame, columns: Sequence[str], max_rows: int = 20) -> str:
    if frame is None or frame.empty:
        return "_No rows available._"
    cols = [c for c in columns if c in frame.columns]
    if not cols:
        return "_No requested columns available._"
    return frame[cols].head(max_rows).to_string(index=False)


def safe_read_csv(path: Path, description: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        print(f"Missing {description}: {path}")
        return pd.DataFrame()
    try:
        frame = pd.read_csv(path)
        print(f"Loaded {description}: {path} shape={frame.shape}")
        return frame
    except Exception as exc:
        print(f"Failed to read {description}: {path}\n  {type(exc).__name__}: {exc}")
        return pd.DataFrame()


def safe_read_parquet(path: Path, description: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        print(f"Missing {description}: {path}")
        return pd.DataFrame()
    try:
        frame = pd.read_parquet(path)
        print(f"Loaded {description}: {path} shape={frame.shape}")
        return frame
    except Exception as exc:
        print(f"Failed to read {description}: {path}\n  {type(exc).__name__}: {exc}")
        return pd.DataFrame()


def safe_read_json(path: Path, description: str) -> Dict[str, Any]:
    path = Path(path)
    if not path.exists():
        print(f"Missing {description}: {path}")
        return {}
    try:
        with path.open("r", encoding="utf-8") as file:
            payload = json.load(file)
        print(f"Loaded {description}: {path}")
        return payload
    except Exception as exc:
        print(f"Failed to read {description}: {path}\n  {type(exc).__name__}: {exc}")
        return {}


def safe_to_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def safe_to_parquet(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    clean = frame.copy()
    for col in clean.columns:
        if pd.api.types.is_object_dtype(clean[col]) or pd.api.types.is_string_dtype(clean[col]):
            clean[col] = clean[col].astype("string")
    clean.to_parquet(path, index=False)


def exp_dir(run_name: str) -> Path:
    return cfg.data_root / "10_experiments" / run_name / cfg.scope


def load_first_existing_csv(directory: Path, filenames: Sequence[str], description: str) -> Tuple[pd.DataFrame, Optional[Path]]:
    for filename in filenames:
        path = directory / filename
        if path.exists():
            return safe_read_csv(path, description), path
    print(f"No CSV found for {description} in {directory}")
    return pd.DataFrame(), None

## 4. Artifact registry

The registry lists expected result files from all previous experiment stages. Missing files are acceptable; they are recorded in `artifact_inventory.csv`.

In [4]:
@dataclass(frozen=True)
class ExperimentSpec:
    experiment_id: str
    description: str
    directory: Path
    candidate_files: Tuple[str, ...]


experiment_specs = [
    ExperimentSpec("m0_temporal", "M0 temporal baselines", exp_dir("m0_baselines_v1_notebook"), ("leaderboard_test.csv", "metrics.csv", "run_config.json")),
    ExperimentSpec("m05_temporal", "M0.5 hybrid LightGBM baselines", exp_dir("m05_hybrid_baselines_v1_notebook"), ("leaderboard_test.csv", "metrics.csv", "run_config.json", "model_notes.json")),
    ExperimentSpec("mlp_temporal", "MLP residual baselines", exp_dir("mlp_residual_v1_notebook"), ("leaderboard_test.csv", "metrics.csv")),
    ExperimentSpec("m05_vs_mlp", "M0.5 vs MLP error analysis", exp_dir("m05_vs_mlp_error_analysis_v1_notebook"), ("leaderboard_test_combined.csv", "paired_summary_test.csv")),
    ExperimentSpec("cold_od_non_graph", "Cold-OD non-graph baselines", exp_dir("cold_od_split_baselines_v1_notebook"), ("leaderboard_test_cold_od.csv", "leaderboard_test.csv", "metrics.csv", "predictions_cold_od_val_test.parquet", "predictions_cold_od_all_splits.parquet")),
    ExperimentSpec("graphv2_cold_od", "GraphSAGE/HGT Cold-OD v2", exp_dir("graphsage_hgt_cold_od_baselines_v2_notebook"), ("leaderboard_test_graph_cold_od_v2.csv", "metrics_graph_cold_od_seed_ensemble_v2.csv", "metrics_graph_cold_od_combined_v2.csv", "combined_predictions_graph_cold_od_val_test_v2.parquet")),
    ExperimentSpec("graph_ablation_v1", "Graph Cold-OD ablation v1", exp_dir("graph_cold_od_ablation_v1_notebook"), ("leaderboard_test_graph_cold_od_ablation.csv", "metrics_graph_cold_od_ablation_seed_ensemble.csv", "paired_summary_graph_cold_od_ablation_test_only.csv")),
    ExperimentSpec("graph_ablation_v2_controls", "Graph Cold-OD ablation v2 controls", exp_dir("graph_cold_od_ablation_v2_controls_notebook"), ("leaderboard_test_graph_cold_od_ablation_v2_controls.csv", "metrics_graph_cold_od_ablation_v2_controls_seed_ensemble.csv")),
    ExperimentSpec("topology_v3", "D-CQHGT topology ablation v3", exp_dir("dcqhgt_topology_cold_od_v3_ablation_notebook"), ("leaderboard_test_dcqhgt_topology_v3_ablation.csv", "metrics_dcqhgt_topology_v3_ablation_seed_ensemble.csv", "metrics_dcqhgt_topology_v3_ablation_combined.csv")),
    ExperimentSpec("disruption_features", "OD-year disruption feature build", exp_dir("build_disruption_features_od_year_v1"), ("tables/disruption_feature_qa_summary.csv", "tables/disruption_source_qa.csv", "tables/global_year_disruption_features.csv")),
    ExperimentSpec("disruption_stabilized", "Disruption gate stabilization", exp_dir("dcqhgt_disruption_gate_stabilization_v1_notebook"), ("leaderboard_test_disruption_gate_stabilized.csv", "metrics_disruption_gate_stabilized_seed_ensemble.csv", "metrics_disruption_gate_stabilized_combined.csv", "combined_predictions_disruption_gate_stabilized.parquet")),
    ExperimentSpec("calibration_cvar", "Calibration and decision-aware CVaR evaluation", exp_dir("calibration_cvar_evaluation_v1_notebook"), ("leaderboard_test_selected_calibration.csv", "metrics_selected_calibration.csv", "decision_cvar_leaderboard_top10pct.csv", "selected_calibrated_predictions_val_test.parquet")),
]

inventory_rows = []
for spec in experiment_specs:
    for rel_path in spec.candidate_files:
        path = spec.directory / rel_path
        inventory_rows.append({
            "experiment_id": spec.experiment_id,
            "description": spec.description,
            "expected_file": rel_path,
            "path": str(path),
            "exists": bool(path.exists()),
            "size_bytes": int(path.stat().st_size) if path.exists() else 0,
        })

artifact_inventory = pd.DataFrame(inventory_rows)
safe_to_csv(artifact_inventory, paths.tables_dir / "artifact_inventory.csv")
print_frame("Artifact inventory", artifact_inventory, max_rows=100)


Artifact inventory
             experiment_id                                    description                                                expected_file                                                                                                                                                                              path  exists  size_bytes
               m0_temporal                          M0 temporal baselines                                         leaderboard_test.csv                                                            E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\leaderboard_test.csv    True        1942
               m0_temporal                          M0 temporal baselines                                                  metrics.csv                                                                     E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\metrics.csv  

## 5. Load dataset and feature metadata

This cell summarizes the task: OD-year rows, features, labels, splits, topology features, and known graph-input paths.

In [5]:
if paths.supervised_with_disruption_path.exists():
    supervised_df = safe_read_parquet(paths.supervised_with_disruption_path, "supervised table with disruption features")
else:
    supervised_df = safe_read_parquet(paths.supervised_model_ready_path, "model-ready supervised table")

feature_manifest = safe_read_json(paths.manifest_path, "feature manifest")
topology_features_df = safe_read_parquet(paths.topology_features_path, "topology feature table")

dataset_rows = []

if not supervised_df.empty:
    if "row_id" not in supervised_df.columns:
        supervised_df = supervised_df.copy()
        supervised_df["row_id"] = np.arange(len(supervised_df), dtype=np.int64)

    dataset_rows.append({"item": "supervised_rows", "value": len(supervised_df)})
    for col in ["faf_orig", "faf_orig_str"]:
        if col in supervised_df.columns:
            dataset_rows.append({"item": "unique_origins", "value": supervised_df[col].nunique()})
            break
    for col in ["faf_dest", "faf_dest_str"]:
        if col in supervised_df.columns:
            dataset_rows.append({"item": "unique_destinations", "value": supervised_df[col].nunique()})
            break
    for col in ["year", "year_int"]:
        if col in supervised_df.columns:
            years = pd.to_numeric(supervised_df[col], errors="coerce")
            dataset_rows.append({"item": "year_min", "value": years.min()})
            dataset_rows.append({"item": "year_max", "value": years.max()})
            break

    for split_col in ["temporal_split", "cold_split"]:
        if split_col in supervised_df.columns:
            for key, value in supervised_df[split_col].value_counts(dropna=False).to_dict().items():
                dataset_rows.append({"item": f"{split_col}_{key}_rows", "value": value})

if feature_manifest:
    manifest_features = feature_manifest.get("feature_columns") or feature_manifest.get("manifest_feature_columns") or []
    historical_features = feature_manifest.get("historical_prior_feature_columns", [])
    label_columns = feature_manifest.get("label_columns", [])
    dataset_rows.append({"item": "manifest_feature_count", "value": len(manifest_features)})
    dataset_rows.append({"item": "historical_prior_feature_count", "value": len(historical_features)})
    dataset_rows.append({"item": "label_columns", "value": ", ".join(label_columns)})

if not topology_features_df.empty:
    topology_cols = [c for c in topology_features_df.columns if c.startswith("topo_")]
    dataset_rows.append({"item": "topology_feature_rows", "value": len(topology_features_df)})
    dataset_rows.append({"item": "topology_feature_count", "value": len(topology_cols)})

dataset_summary = pd.DataFrame(dataset_rows)
safe_to_csv(dataset_summary, paths.tables_dir / "table_1_dataset_summary.csv")
print_frame("Table 1: Dataset summary", dataset_summary, max_rows=80)

Loaded supervised table with disruption features: E:\NetworkOptimization\pythonProject1\Data\08_processed\disruption_features\freight_mnet_supervised_edges_2018_2024_east_plus_gulf_with_disruption.parquet shape=(73972, 219)
Loaded feature manifest: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json
Loaded topology feature table: E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs\topology_features_od_east_plus_gulf.parquet shape=(73972, 22)

Table 1: Dataset summary
                          item                           value
               supervised_rows                           73972
                unique_origins                             104
           unique_destinations                             104
                      year_min                            2018
                      year_max                            2024
        manifest_feature_count                              6

## 6. Metric normalization functions

These functions standardize result tables from many notebooks. They also map long internal model names into concise paper-facing names.

In [6]:
METRIC_RENAME_MAP = {
    "q25_mae": "mae_q25",
    "q50_mae": "mae_q50",
    "q75_mae": "mae_q75",
    "mae_iqr": "iqr_mae",
    "mean_pinball": "pinball_mean",
    "pinball": "pinball_mean",
    "weighted_pinball": "weighted_pinball_mean",
    "top_q75_10_mae_q75": "stress_top10_mae_q75",
}

COMMON_METRIC_COLUMNS = [
    "pinball_mean",
    "weighted_pinball_mean",
    "mae_q50",
    "mae_q75",
    "iqr_mae",
    "stress_top10_mae_q75",
    "sparse_bottom25_mae_q75",
    "top_iqr10_mae_q75",
    "top_iqr10_iqr_mae",
]


def assign_paper_model_name(row: pd.Series) -> str:
    text = " ".join([
        str(row.get("experiment_id", "")),
        str(row.get("source", "")),
        str(row.get("model", "")),
        str(row.get("checkpoint_metric", "")),
        str(row.get("calibration_method", "")),
        str(row.get("disruption_group", "")),
    ]).lower()

    if "graphv2_hgt" in text or ("hgt_residual_prior" in text and "reference" in text):
        return "HGT-ColdOD"
    if "graphv2_graphsage" in text or ("graphsage_residual_prior" in text and "reference" in text):
        return "GraphSAGE-ColdOD"
    if "graph_ensemble" in text and "hgt_residual_prior" in text:
        return "HGT-ColdOD-Ensemble"
    if "graph_ensemble" in text and "graphsage_residual_prior" in text:
        return "GraphSAGE-ColdOD-Ensemble"
    if "mlp_residual_prior" in text:
        return "MLP-Residual"
    if "prior_feature_lgbm_direct" in text:
        return "HistPrior-Feature-LGBM"
    if "residual_lgbm_prior" in text:
        return "HistPrior-Residual-LGBM"
    if "od_historical" in text:
        return "HistPrior"
    if "global_train_median" in text or "global median" in text:
        return "GlobalMedian"
    if "lightgbm_direct" in text:
        return "LGBM-Direct"
    if "topology_features_only" in text:
        return "HGT-TopoFeatures"
    if "truck_rail_plus_topology_features" in text:
        return "HGT-TruckRail+TopoFeatures"
    if "terminal_plus_truck_plus_rail" in text:
        return "HGT-Terminal+TruckRail"
    if "full_topology" in text:
        return "D-CQHGT-Topology"
    if "weather_zone" in text:
        return "D-CQHGT-Weather"
    return str(row.get("model", "unknown"))


def normalize_metrics_table(frame: pd.DataFrame, experiment_id: str, source_label: str) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()

    clean = frame.copy()
    clean = clean.rename(columns={k: v for k, v in METRIC_RENAME_MAP.items() if k in clean.columns})

    if "source" not in clean.columns:
        clean["source"] = source_label
    if "model" not in clean.columns:
        clean["model"] = clean.get("candidate_model", "unknown")
    if "checkpoint_metric" not in clean.columns:
        if "checkpoint" in clean.columns:
            clean["checkpoint_metric"] = clean["checkpoint"]
        elif "calibration_method" in clean.columns:
            clean["checkpoint_metric"] = clean["calibration_method"]
        else:
            clean["checkpoint_metric"] = "baseline"
    if "seed" not in clean.columns:
        clean["seed"] = "baseline"

    clean["experiment_id"] = experiment_id
    clean["source_label"] = source_label

    for col in COMMON_METRIC_COLUMNS:
        if col in clean.columns:
            clean[col] = pd.to_numeric(clean[col], errors="coerce")

    if "pinball_mean" in clean.columns and "rank" not in clean.columns:
        clean["rank"] = clean["pinball_mean"].rank(method="first", ascending=True).astype(int)

    clean["paper_model"] = clean.apply(assign_paper_model_name, axis=1)
    clean["model_key"] = (
        clean["source"].astype(str) + "::"
        + clean["model"].astype(str) + "::"
        + clean["checkpoint_metric"].astype(str) + "::"
        + clean["seed"].astype(str)
    )

    return clean

## 7. Load all available metric tables

Each row is tagged with its experiment ID so that later cells can build topic-specific paper tables.

In [7]:
metric_load_specs = [
    ("m0_temporal", "M0 temporal", exp_dir("m0_baselines_v1_notebook"), ["leaderboard_test.csv", "metrics.csv"]),
    ("m05_temporal", "M0.5 temporal", exp_dir("m05_hybrid_baselines_v1_notebook"), ["leaderboard_test.csv", "metrics.csv"]),
    ("mlp_temporal", "MLP temporal", exp_dir("mlp_residual_v1_notebook"), ["leaderboard_test_mlp_residual.csv", "leaderboard_test.csv", "metrics.csv"]),
    ("m05_vs_mlp", "M0.5 vs MLP combined", exp_dir("m05_vs_mlp_error_analysis_v1_notebook"), ["leaderboard_test_combined.csv"]),
    ("cold_od_non_graph", "Cold-OD non-graph", exp_dir("cold_od_split_baselines_v1_notebook"), ["leaderboard_test_cold_od.csv", "leaderboard_test.csv", "cold_od_recomputed_leaderboard_test.csv"]),
    ("graphv2_cold_od", "GraphV2 Cold-OD", exp_dir("graphsage_hgt_cold_od_baselines_v2_notebook"), ["leaderboard_test_graph_cold_od_v2.csv", "metrics_graph_cold_od_combined_v2.csv"]),
    ("graph_ablation_v1", "Graph ablation v1", exp_dir("graph_cold_od_ablation_v1_notebook"), ["leaderboard_test_graph_cold_od_ablation.csv", "metrics_graph_cold_od_ablation_seed_ensemble.csv"]),
    ("graph_ablation_v2_controls", "Graph ablation v2 controls", exp_dir("graph_cold_od_ablation_v2_controls_notebook"), ["leaderboard_test_graph_cold_od_ablation_v2_controls.csv", "metrics_graph_cold_od_ablation_v2_controls_seed_ensemble.csv"]),
    ("topology_v3", "Topology ablation v3", exp_dir("dcqhgt_topology_cold_od_v3_ablation_notebook"), ["leaderboard_test_dcqhgt_topology_v3_ablation.csv", "metrics_dcqhgt_topology_v3_ablation_seed_ensemble.csv"]),
    ("disruption_stabilized", "Disruption stabilized", exp_dir("dcqhgt_disruption_gate_stabilization_v1_notebook"), ["leaderboard_test_disruption_gate_stabilized.csv", "metrics_disruption_gate_stabilized_seed_ensemble.csv"]),
    ("calibration_cvar", "Selected calibration", exp_dir("calibration_cvar_evaluation_v1_notebook"), ["leaderboard_test_selected_calibration.csv", "metrics_selected_calibration.csv"]),
]

metric_frames = []
metric_source_rows = []

for experiment_id, source_label, directory, filenames in metric_load_specs:
    frame, loaded_path = load_first_existing_csv(directory, filenames, source_label)
    if not frame.empty:
        normalized = normalize_metrics_table(frame, experiment_id=experiment_id, source_label=source_label)
        metric_frames.append(normalized)
        metric_source_rows.append({
            "experiment_id": experiment_id,
            "source_label": source_label,
            "loaded_path": str(loaded_path),
            "n_rows": len(frame),
            "n_columns": frame.shape[1],
        })

all_metrics = pd.concat(metric_frames, ignore_index=True) if metric_frames else pd.DataFrame()
metric_sources = pd.DataFrame(metric_source_rows)

safe_to_csv(metric_sources, paths.tables_dir / "metric_source_inventory.csv")
safe_to_csv(all_metrics, paths.tables_dir / "all_loaded_metrics_normalized.csv")

print_frame("Metric source inventory", metric_sources, max_rows=50)
print_frame("All loaded metrics normalized", all_metrics, max_rows=12)

Loaded M0 temporal: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m0_baselines_v1_notebook\east_plus_gulf\leaderboard_test.csv shape=(10, 12)
Loaded M0.5 temporal: E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_hybrid_baselines_v1_notebook\east_plus_gulf\leaderboard_test.csv shape=(8, 38)
Loaded MLP temporal: E:\NetworkOptimization\pythonProject1\Data\10_experiments\mlp_residual_v1_notebook\east_plus_gulf\leaderboard_test.csv shape=(6, 34)
No CSV found for M0.5 vs MLP combined in E:\NetworkOptimization\pythonProject1\Data\10_experiments\m05_vs_mlp_error_analysis_v1_notebook\east_plus_gulf
Loaded Cold-OD non-graph: E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\leaderboard_test.csv shape=(11, 32)
Loaded GraphV2 Cold-OD: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graphsage_hgt_cold_od_baselines_v2_notebook\east_plus_gulf\leaderboard_test_graph_cold_od_v2.csv shape=(47, 32)
Loaded Graph

## 8. Helper for selecting representative rows

For each paper table, we select the best available row for a named model family. Ensemble rows are preferred where relevant.

In [8]:
def select_best_by_terms(
    frame: pd.DataFrame,
    experiment_ids: Sequence[str],
    label: str,
    include_terms: Sequence[str],
    prefer_ensemble: bool = True,
    exclude_terms: Sequence[str] = (),
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()

    subset = frame.loc[frame["experiment_id"].isin(experiment_ids)].copy()
    if subset.empty:
        return pd.DataFrame()

    text = (
        subset["source"].astype(str) + " "
        + subset["model"].astype(str) + " "
        + subset["paper_model"].astype(str) + " "
        + subset["checkpoint_metric"].astype(str)
    ).str.lower()

    mask = pd.Series(True, index=subset.index)
    for term in include_terms:
        mask &= text.str.contains(term.lower(), na=False)
    for term in exclude_terms:
        mask &= ~text.str.contains(term.lower(), na=False)

    subset = subset.loc[mask].copy()
    if subset.empty or "pinball_mean" not in subset.columns:
        return pd.DataFrame()

    if prefer_ensemble:
        subset["_ensemble_priority"] = subset["seed"].astype(str).eq("ensemble").map({True: 0, False: 1})
    else:
        subset["_ensemble_priority"] = 0

    subset = subset.sort_values(["_ensemble_priority", "pinball_mean"], ascending=[True, True])
    best = subset.head(1).drop(columns=["_ensemble_priority"])
    best["paper_table_group"] = label
    return best


def make_table_from_patterns(frame: pd.DataFrame, patterns: Sequence[Tuple[str, Sequence[str], Sequence[str], Sequence[str]]], prefer_ensemble: bool = True) -> pd.DataFrame:
    rows = []
    for label, experiment_ids, include_terms, exclude_terms in patterns:
        best = select_best_by_terms(frame, experiment_ids, label, include_terms, prefer_ensemble=prefer_ensemble, exclude_terms=exclude_terms)
        if not best.empty:
            rows.append(best)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


DISPLAY_METRIC_COLUMNS = [
    "paper_table_group",
    "paper_model",
    "source",
    "model",
    "checkpoint_metric",
    "seed",
    "pinball_mean",
    "weighted_pinball_mean",
    "mae_q50",
    "mae_q75",
    "iqr_mae",
    "stress_top10_mae_q75",
    "sparse_bottom25_mae_q75",
    "experiment_id",
]

## 9. Final paper tables

The following cells produce the paper-ready tables for dataset summary, temporal baselines, Cold-OD main results, graph ablations, topology, disruption, calibration, and CVaR screening.

In [9]:
# Table 2: Temporal baselines.
temporal_patterns = [
    ("GlobalMedian", ["m0_temporal", "m05_temporal"], ["global"], []),
    ("HistPrior", ["m0_temporal", "m05_temporal"], ["historical"], []),
    ("LGBM-Direct", ["m0_temporal", "m05_temporal"], ["lightgbm_direct"], []),
    ("HistPrior-Residual-LGBM", ["m05_temporal", "m05_vs_mlp"], ["residual_lgbm_prior"], []),
    ("HistPrior-Feature-LGBM", ["m05_temporal", "m05_vs_mlp"], ["prior_feature_lgbm_direct"], []),
    ("MLP-Residual", ["mlp_temporal", "m05_vs_mlp"], ["mlp_residual_prior"], []),
]
table_temporal = make_table_from_patterns(all_metrics, temporal_patterns, prefer_ensemble=False)
safe_to_csv(table_temporal, paths.tables_dir / "table_2_temporal_baselines.csv")
print_frame("Table 2: Temporal baselines", table_temporal[[c for c in DISPLAY_METRIC_COLUMNS if c in table_temporal.columns]], max_rows=20)

# Table 3: Cold-OD main results.
cold_patterns = [
    ("ColdOD-MLP", ["cold_od_non_graph", "calibration_cvar"], ["mlp_residual_prior"], []),
    ("HGT-ColdOD", ["graphv2_cold_od", "calibration_cvar"], ["graphv2_hgt"], []),
    ("GraphSAGE-ColdOD", ["graphv2_cold_od", "calibration_cvar"], ["graphv2_graphsage"], []),
    ("HGT-ColdOD-Ensemble", ["graphv2_cold_od", "calibration_cvar"], ["graph_ensemble", "hgt_residual_prior"], []),
    ("GraphSAGE-ColdOD-Ensemble", ["graphv2_cold_od", "calibration_cvar"], ["graph_ensemble", "graphsage_residual_prior"], []),
]
table_cold_od = make_table_from_patterns(all_metrics, cold_patterns, prefer_ensemble=True)
safe_to_csv(table_cold_od, paths.tables_dir / "table_3_cold_od_main_results.csv")
print_frame("Table 3: Cold-OD main results", table_cold_od[[c for c in DISPLAY_METRIC_COLUMNS if c in table_cold_od.columns]], max_rows=20)


Table 2: Temporal baselines
      paper_table_group               paper_model        source                        model checkpoint_metric     seed  pinball_mean  weighted_pinball_mean    mae_q50    mae_q75    iqr_mae  stress_top10_mae_q75  sparse_bottom25_mae_q75 experiment_id
           GlobalMedian hist_lgbm_ensemble_global M0.5 temporal    hist_lgbm_ensemble_global          baseline baseline    146.728685             130.655899 231.257504 462.529352 370.897768            694.091366               562.211623  m05_temporal
              HistPrior                 HistPrior   M0 temporal         od_historical_median          baseline baseline    153.000149                    NaN 230.534636 472.672178 385.553897            709.868526               577.776247   m0_temporal
            LGBM-Direct               LGBM-Direct M0.5 temporal              lightgbm_direct          baseline baseline    157.521557             140.907413 283.130157 571.960199 521.059845            690.011190       

In [10]:
# Table 4: Graph ablation.
graph_ablation_patterns = [
    ("NoGraph-MLP", ["graph_ablation_v1", "graph_ablation_v2_controls"], ["mlp_same_features_no_graph"], []),
    ("SelfLoopOnly", ["graph_ablation_v1", "graph_ablation_v2_controls"], ["self_loop"], []),
    ("TrainODOnly", ["graph_ablation_v1", "graph_ablation_v2_controls"], ["train_od_only"], []),
    ("DemandEdgesOnly", ["graph_ablation_v1", "graph_ablation_v2_controls"], ["demand_edges_only"], []),
    ("FullFAFGraph", ["graph_ablation_v1", "graph_ablation_v2_controls"], ["full_faf_zone_graph"], []),
    ("GraphSAGE", ["graph_ablation_v1", "graph_ablation_v2_controls"], ["graphsage_same_edges"], []),
    ("ShuffledEdges", ["graph_ablation_v1", "graph_ablation_v2_controls"], ["shuffled_graph_edges"], []),
    ("RandomizedEdgeTypes", ["graph_ablation_v1", "graph_ablation_v2_controls"], ["randomized_edge_types"], []),
]
table_graph_ablation = make_table_from_patterns(all_metrics, graph_ablation_patterns, prefer_ensemble=True)
safe_to_csv(table_graph_ablation, paths.tables_dir / "table_4_graph_ablation.csv")
print_frame("Table 4: Graph ablation", table_graph_ablation[[c for c in DISPLAY_METRIC_COLUMNS if c in table_graph_ablation.columns]], max_rows=30)

# Table 5: Topology ablation.
topology_patterns = [
    ("NoTopology-HGT", ["topology_v3"], ["faf_hgt_no_topology"], []),
    ("TopologyFeaturesOnly", ["topology_v3"], ["topology_features_only"], []),
    ("TruckAdjOnly", ["topology_v3"], ["truck_adj_only"], []),
    ("RailAdjOnly", ["topology_v3"], ["rail_adj_only"], []),
    ("TruckPlusRail", ["topology_v3"], ["truck_plus_rail_adj"], []),
    ("TerminalAccessOnly", ["topology_v3"], ["terminal_access_only"], []),
    ("TerminalPlusTruckRail", ["topology_v3"], ["terminal_plus_truck_plus_rail"], []),
    ("TruckRailPlusTopoFeatures", ["topology_v3"], ["truck_rail_plus_topology_features"], []),
    ("FullTopologyNoDisruption", ["topology_v3"], ["full_topology_no_disruption"], []),
]
table_topology = make_table_from_patterns(all_metrics, topology_patterns, prefer_ensemble=True)
safe_to_csv(table_topology, paths.tables_dir / "table_5_topology_ablation.csv")
print_frame("Table 5: Topology ablation", table_topology[[c for c in DISPLAY_METRIC_COLUMNS if c in table_topology.columns]], max_rows=30)


Table 4: Graph ablation
  paper_table_group                  paper_model               source                        model checkpoint_metric     seed  pinball_mean  weighted_pinball_mean    mae_q50    mae_q75    iqr_mae  stress_top10_mae_q75  sparse_bottom25_mae_q75              experiment_id
        NoGraph-MLP   mlp_same_features_no_graph ABLATION_V2_ENSEMBLE   mlp_same_features_no_graph  best_val_pinball ensemble    218.555618             203.794327 405.917755 748.611816 578.086304            900.295898                      NaN graph_ablation_v2_controls
       SelfLoopOnly           hgt_self_loop_only ABLATION_V2_ENSEMBLE           hgt_self_loop_only  best_val_pinball ensemble    130.370346             117.928032 223.665863 441.030182 370.140503            523.466980                      NaN graph_ablation_v2_controls
        TrainODOnly            hgt_train_od_only    ABLATION_ENSEMBLE            hgt_train_od_only  best_val_pinball ensemble    130.845367             118.525673 22

In [11]:
# Table 6: Disruption ablation.
disruption_patterns = [
    ("NoDisruption-Reproduced", ["disruption_stabilized", "calibration_cvar"], ["no_disruption"], []),
    ("WeatherZoneOnly", ["disruption_stabilized", "calibration_cvar"], ["weather_zone_only"], []),
    ("BorderGlobalOnly", ["disruption_stabilized", "calibration_cvar"], ["border_global_only"], []),
    ("WeatherPlusBorder", ["disruption_stabilized", "calibration_cvar"], ["weather_zone_plus_border"], []),
    ("FullDisruption", ["disruption_stabilized", "calibration_cvar"], ["full_disruption"], []),
]
table_disruption = make_table_from_patterns(all_metrics, disruption_patterns, prefer_ensemble=True)
safe_to_csv(table_disruption, paths.tables_dir / "table_6_disruption_ablation.csv")
print_frame("Table 6: Disruption ablation", table_disruption[[c for c in DISPLAY_METRIC_COLUMNS if c in table_disruption.columns]], max_rows=30)

# Table 7: Calibration.
calibration_leaderboard = safe_read_csv(exp_dir("calibration_cvar_evaluation_v1_notebook") / "leaderboard_test_selected_calibration.csv", "selected calibration leaderboard")
if not calibration_leaderboard.empty:
    table_calibration = normalize_metrics_table(calibration_leaderboard, "calibration_cvar", "Calibration")
    table_calibration = table_calibration.sort_values("pinball_mean", ascending=True).head(20)
else:
    table_calibration = pd.DataFrame()
safe_to_csv(table_calibration, paths.tables_dir / "table_7_calibration_summary.csv")
print_frame("Table 7: Calibration summary", table_calibration[[c for c in ["paper_model", "source", "model", "calibration_method", "pinball_mean", "mae_q75", "iqr_mae"] if c in table_calibration.columns]], max_rows=20)

# Table 8: Decision-aware CVaR screening.
decision_cvar = safe_read_csv(exp_dir("calibration_cvar_evaluation_v1_notebook") / "decision_cvar_leaderboard_top10pct.csv", "decision CVaR top-10% leaderboard")
if not decision_cvar.empty:
    sort_col = "cvar_screening_regret" if "cvar_screening_regret" in decision_cvar.columns else ("normalized_cvar_regret" if "normalized_cvar_regret" in decision_cvar.columns else None)
    if sort_col:
        decision_cvar[sort_col] = pd.to_numeric(decision_cvar[sort_col], errors="coerce")
        table_cvar = decision_cvar.sort_values(sort_col, ascending=True).head(20)
    else:
        table_cvar = decision_cvar.head(20)
else:
    table_cvar = pd.DataFrame()
safe_to_csv(table_cvar, paths.tables_dir / "table_8_decision_cvar_screening.csv")
print_frame("Table 8: Decision-aware CVaR screening", table_cvar, max_rows=20)


Table 6: Disruption ablation
(empty)
Loaded selected calibration leaderboard: E:\NetworkOptimization\pythonProject1\Data\10_experiments\calibration_cvar_evaluation_v1_notebook\east_plus_gulf\leaderboard_test_selected_calibration.csv shape=(69, 26)

Table 7: Calibration summary
                paper_model                         source                                 model calibration_method  pinball_mean    mae_q75    iqr_mae
                 HGT-ColdOD                      REFERENCE                           graphv2_hgt               none    128.830055 453.218931 377.083895
        HGT-ColdOD-Ensemble                 GRAPH_ENSEMBLE           hgt_residual_prior_features               none    130.042694 453.410270 377.061278
           GraphSAGE-ColdOD                      REFERENCE                     graphv2_graphsage               none    130.804053 434.024788 359.878583
  GraphSAGE-ColdOD-Ensemble                 GRAPH_ENSEMBLE     graphsage_residual_prior_features               no

## 10. Repair and regenerate segment summaries

This cell attaches `n_fmi_county_pairs` from the supervised table by `row_id`, preventing the previous `_x/_y` merge bug and ensuring test-only sparse diagnostics.

In [12]:
def build_row_metadata_lookup(supervised: pd.DataFrame) -> pd.DataFrame:
    if supervised is None or supervised.empty:
        return pd.DataFrame(columns=["row_id", "n_fmi_county_pairs"])

    lookup = supervised.copy()
    if "row_id" not in lookup.columns:
        lookup["row_id"] = np.arange(len(lookup), dtype=np.int64)
    if "n_fmi_county_pairs" not in lookup.columns:
        lookup["n_fmi_county_pairs"] = np.nan

    cols = ["row_id", "n_fmi_county_pairs"]
    for col in ["obs_weight_sum", "faf_orig", "faf_dest", "faf_orig_str", "faf_dest_str", "year", "year_int"]:
        if col in lookup.columns:
            cols.append(col)

    lookup = lookup[cols].copy()
    lookup["row_id"] = pd.to_numeric(lookup["row_id"], errors="coerce")
    lookup["n_fmi_county_pairs"] = pd.to_numeric(lookup["n_fmi_county_pairs"], errors="coerce")
    lookup = lookup.dropna(subset=["row_id"]).copy()
    lookup["row_id"] = lookup["row_id"].astype(int)
    return lookup.drop_duplicates("row_id", keep="first")


row_metadata_lookup = build_row_metadata_lookup(supervised_df)
safe_to_csv(row_metadata_lookup, paths.tables_dir / "row_metadata_lookup.csv")


def normalize_prediction_frame(frame: pd.DataFrame, source_label: str) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()

    clean = frame.copy()
    rename = {
        "actual_q25": "true_q25",
        "actual_q50": "true_q50",
        "actual_q75": "true_q75",
        "y_q25": "true_q25",
        "y_q50": "true_q50",
        "y_q75": "true_q75",
        "truck_q25": "true_q25",
        "truck_q50": "true_q50",
        "truck_q75": "true_q75",
        "prediction_q25": "pred_q25",
        "prediction_q50": "pred_q50",
        "prediction_q75": "pred_q75",
        "q25_pred": "pred_q25",
        "q50_pred": "pred_q50",
        "q75_pred": "pred_q75",
        "predicted_q25": "pred_q25",
        "predicted_q50": "pred_q50",
        "predicted_q75": "pred_q75",
        "cold_split": "split",
    }
    clean = clean.rename(columns={k: v for k, v in rename.items() if k in clean.columns})

    if "source" not in clean.columns:
        clean["source"] = source_label
    if "model" not in clean.columns:
        clean["model"] = "unknown"
    if "checkpoint_metric" not in clean.columns:
        clean["checkpoint_metric"] = clean["checkpoint"] if "checkpoint" in clean.columns else ("calibration_method" if "calibration_method" in clean.columns else "baseline")
    if "seed" not in clean.columns:
        clean["seed"] = "baseline"
    if "split" not in clean.columns:
        raise ValueError(f"{source_label} prediction table is missing split/cold_split.")
    if "row_id" not in clean.columns:
        raise ValueError(f"{source_label} prediction table is missing row_id. Use row-id reconstruction notebook first.")

    clean["row_id"] = pd.to_numeric(clean["row_id"], errors="raise").astype(int)
    clean["split"] = clean["split"].astype(str).str.lower()

    # Attach row-level metadata. Use suffixes and then coalesce.
    clean = clean.merge(row_metadata_lookup[["row_id", "n_fmi_county_pairs"]], on="row_id", how="left", suffixes=("", "_lookup"), validate="many_to_one")
    n_fmi_cols = [c for c in ["n_fmi_county_pairs", "n_fmi_county_pairs_lookup", "n_fmi_county_pairs_x", "n_fmi_county_pairs_y"] if c in clean.columns]
    if n_fmi_cols:
        series = clean[n_fmi_cols[0]]
        for col in n_fmi_cols[1:]:
            series = series.combine_first(clean[col])
        clean["n_fmi_county_pairs"] = pd.to_numeric(series, errors="coerce")
        clean = clean.drop(columns=[c for c in n_fmi_cols if c != "n_fmi_county_pairs"], errors="ignore")

    # Attach true labels if missing.
    label_map = {"true_q25": "truck_q25", "true_q50": "truck_q50", "true_q75": "truck_q75"}
    if any(k not in clean.columns for k in label_map):
        label_cols = ["row_id"] + [v for v in label_map.values() if v in supervised_df.columns]
        labels = supervised_df[label_cols].copy()
        labels = labels.rename(columns={v: k for k, v in label_map.items() if v in labels.columns})
        clean = clean.merge(labels, on="row_id", how="left", validate="many_to_one", suffixes=("", "_label"))

    for col in ["true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75"]:
        if col in clean.columns:
            clean[col] = pd.to_numeric(clean[col], errors="coerce")

    return clean


prediction_sources = [
    ("GraphV2", exp_dir("graphsage_hgt_cold_od_baselines_v2_notebook") / "combined_predictions_graph_cold_od_val_test_v2.parquet"),
    ("DisruptionStabilized", exp_dir("dcqhgt_disruption_gate_stabilization_v1_notebook") / "combined_predictions_disruption_gate_stabilized.parquet"),
    ("CalibrationSelected", exp_dir("calibration_cvar_evaluation_v1_notebook") / "selected_calibrated_predictions_val_test.parquet"),
]

prediction_frames = []
for source_label, path in prediction_sources:
    raw = safe_read_parquet(path, f"{source_label} prediction table")
    if not raw.empty:
        try:
            prediction_frames.append(normalize_prediction_frame(raw, source_label))
        except Exception as exc:
            print(f"Skipping {source_label} prediction table: {type(exc).__name__}: {exc}")

final_predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
safe_to_parquet(final_predictions, paths.output_dir / "normalized_final_predictions_val_test.parquet")
print_frame("Normalized final predictions", final_predictions, max_rows=12)

Loaded GraphV2 prediction table: E:\NetworkOptimization\pythonProject1\Data\10_experiments\graphsage_hgt_cold_od_baselines_v2_notebook\east_plus_gulf\combined_predictions_graph_cold_od_val_test_v2.parquet shape=(94658, 31)
Skipping GraphV2 prediction table: ValueError: GraphV2 prediction table is missing row_id. Use row-id reconstruction notebook first.
Loaded DisruptionStabilized prediction table: E:\NetworkOptimization\pythonProject1\Data\10_experiments\dcqhgt_disruption_gate_stabilization_v1_notebook\east_plus_gulf\combined_predictions_disruption_gate_stabilized.parquet shape=(19283, 40)
Loaded CalibrationSelected prediction table: E:\NetworkOptimization\pythonProject1\Data\10_experiments\calibration_cvar_evaluation_v1_notebook\east_plus_gulf\selected_calibrated_predictions_val_test.parquet shape=(136095, 45)

Normalized final predictions
   source                     model       variant split faf_orig faf_dest  od_pair   year  true_q25  true_q50  true_q75    pred_q25    pred_q50   

In [13]:
TAUS = np.array([0.25, 0.50, 0.75], dtype=float)


def pinball_loss_np(y_true: np.ndarray, y_pred: np.ndarray, taus: np.ndarray = TAUS) -> np.ndarray:
    errors = y_true - y_pred
    losses = np.maximum(taus * errors, (taus - 1.0) * errors)
    return losses.mean(axis=1)


def compute_prediction_metrics(group: pd.DataFrame) -> Dict[str, float]:
    true = group[["true_q25", "true_q50", "true_q75"]].to_numpy(dtype=float)
    pred = group[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(dtype=float)
    pinball = pinball_loss_np(true, pred)
    return {
        "n_rows": int(len(group)),
        "pinball_mean": float(np.mean(pinball)),
        "mae_q50": float(np.mean(np.abs(true[:, 1] - pred[:, 1]))),
        "mae_q75": float(np.mean(np.abs(true[:, 2] - pred[:, 2]))),
        "iqr_mae": float(np.mean(np.abs((true[:, 2] - true[:, 0]) - (pred[:, 2] - pred[:, 0])))),
    }


def make_quantile_labels(values: pd.Series, q: int, prefix: str) -> pd.Series:
    values = pd.to_numeric(values, errors="coerce")
    output = pd.Series("missing", index=values.index, dtype="object")
    valid = values.notna()
    if valid.sum() == 0:
        return output
    if values.loc[valid].nunique() < 2:
        output.loc[valid] = f"{prefix}_01"
        return output
    try:
        codes = pd.qcut(values.loc[valid], q=q, labels=False, duplicates="drop")
    except ValueError:
        output.loc[valid] = f"{prefix}_01"
        return output
    output.loc[valid] = [f"{prefix}_{int(code) + 1:02d}" for code in codes]
    return output


def add_segment_columns(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    out = frame.copy()
    out["split"] = out["split"].astype(str).str.lower()
    out["true_iqr"] = out["true_q75"] - out["true_q25"]
    test_mask = out["split"].eq("test")

    out["true_q75_decile"] = "not_test"
    out["true_iqr_decile"] = "not_test"
    out["n_fmi_county_pairs_quartile"] = "not_test"

    out.loc[test_mask, "true_q75_decile"] = make_quantile_labels(out.loc[test_mask, "true_q75"], q=10, prefix="true_q75_decile")
    out.loc[test_mask, "true_iqr_decile"] = make_quantile_labels(out.loc[test_mask, "true_iqr"], q=10, prefix="true_iqr_decile")

    if "n_fmi_county_pairs" in out.columns and out.loc[test_mask, "n_fmi_county_pairs"].notna().any():
        out.loc[test_mask, "n_fmi_county_pairs_quartile"] = make_quantile_labels(out.loc[test_mask, "n_fmi_county_pairs"], q=4, prefix="n_fmi_q")
    else:
        out.loc[test_mask, "n_fmi_county_pairs_quartile"] = "missing"

    return out


def segment_summary(frame: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    test = frame.query("split == 'test'").copy()
    if test.empty:
        return pd.DataFrame()

    for col in ["source", "model", "checkpoint_metric", "seed"]:
        if col not in test.columns:
            test[col] = "unknown"
        test[col] = test[col].astype(str)

    rows = []
    group_cols = ["source", "model", "checkpoint_metric", "seed", segment_col]
    for keys, group in test.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys))
        row.update(compute_prediction_metrics(group))
        rows.append(row)
    return pd.DataFrame(rows)


final_predictions_segmented = add_segment_columns(final_predictions)

segment_tables = {}
for segment_col in ["true_q75_decile", "true_iqr_decile", "n_fmi_county_pairs_quartile"]:
    table = segment_summary(final_predictions_segmented, segment_col)
    segment_tables[segment_col] = table
    safe_to_csv(table, paths.tables_dir / f"final_segment_summary__{segment_col}.csv")
    print_frame(f"Final segment summary: {segment_col}", table, max_rows=15)

if not final_predictions_segmented.empty:
    sparse_audit = (
        final_predictions_segmented.query("split == 'test'")
        .groupby(["source", "model", "checkpoint_metric", "seed"], as_index=False)
        .agg(
            n_segments=("n_fmi_county_pairs_quartile", "nunique"),
            total_rows=("row_id", "nunique"),
            nonmissing_n_fmi=("n_fmi_county_pairs", lambda s: int(pd.to_numeric(s, errors="coerce").notna().sum())),
        )
    )
else:
    sparse_audit = pd.DataFrame()

safe_to_csv(sparse_audit, paths.tables_dir / "final_sparse_segment_audit.csv")
print_frame("Final sparse segment audit", sparse_audit, max_rows=30)


Final segment summary: true_q75_decile
source             model checkpoint_metric           seed    true_q75_decile  n_rows  pinball_mean     mae_q50     mae_q75     iqr_mae
ColdOD destination_prior    not_applicable not_applicable true_q75_decile_01     106    846.289144 1801.592809 2652.793556 1352.963370
ColdOD destination_prior    not_applicable not_applicable true_q75_decile_02     106    524.509113 1006.044485 1758.201388  916.928391
ColdOD destination_prior    not_applicable not_applicable true_q75_decile_03     105    337.557005  702.278977 1049.651796  573.150621
ColdOD destination_prior    not_applicable not_applicable true_q75_decile_04     106    201.483331  469.299243  586.526229  480.577611
ColdOD destination_prior    not_applicable not_applicable true_q75_decile_05     106    195.516679  384.340054  556.139895  474.434333
ColdOD destination_prior    not_applicable not_applicable true_q75_decile_06     106    265.807255  460.389793  694.837360  516.363130
ColdOD destinat

## 11. Optional paper plots

The plots use default matplotlib styling and are only generated when matplotlib is available.

In [14]:
def plot_horizontal_leaderboard(frame: pd.DataFrame, value_col: str, label_col: str, title: str, path: Path, top_n: int = 15) -> None:
    if not MATPLOTLIB_AVAILABLE or frame.empty or value_col not in frame.columns or label_col not in frame.columns:
        return
    plot_df = frame.dropna(subset=[value_col]).sort_values(value_col, ascending=True).head(top_n).copy()
    if plot_df.empty:
        return
    plt.figure(figsize=(10, max(5, 0.35 * len(plot_df))))
    plt.barh(plot_df[label_col].astype(str), plot_df[value_col])
    plt.gca().invert_yaxis()
    plt.xlabel(value_col)
    plt.title(title)
    plt.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=180)
    plt.close()


if not table_cold_od.empty:
    table_cold_od = table_cold_od.copy()
    table_cold_od["plot_label"] = table_cold_od["paper_table_group"].astype(str)
    plot_horizontal_leaderboard(
        table_cold_od,
        "pinball_mean",
        "plot_label",
        "Cold-OD Main Results: Test Pinball Mean",
        paths.plots_dir / "cold_od_main_pinball.png",
        top_n=12,
    )

if not table_cvar.empty:
    cvar_col = "cvar_screening_regret" if "cvar_screening_regret" in table_cvar.columns else ("normalized_cvar_regret" if "normalized_cvar_regret" in table_cvar.columns else None)
    if cvar_col:
        label_col = "model" if "model" in table_cvar.columns else table_cvar.columns[0]
        plot_horizontal_leaderboard(
            table_cvar,
            cvar_col,
            label_col,
            "Decision-Aware OD-Risk Screening Regret, Top 10%",
            paths.plots_dir / "decision_cvar_regret_top10pct.png",
            top_n=12,
        )

print("Plot directory:", paths.plots_dir)

Plot directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\final_results_consolidation_v1_notebook\east_plus_gulf\plots


## 12. Final interpretation report

The report is intentionally plain text and does not require `tabulate`.

In [15]:
report_lines = []
report_lines.append("# FREIGHT-MNet Final Results Consolidation Report")
report_lines.append("")
report_lines.append(f"Created at UNIX time: {now_unix():.3f}")
report_lines.append(f"Data root: {cfg.data_root}")
report_lines.append(f"Scope: {cfg.scope}")
report_lines.append("")
report_lines.append("## Executive interpretation")
report_lines.append("")
report_lines.append(
    "The core experimental chain is complete. The strongest empirical result is Cold-OD graph generalization: "
    "GraphV2 HGT is the best overall pinball model, while GraphV2 GraphSAGE is the strongest q75/IQR and decision-aware risk-screening model. "
    "Topology and weather-disruption modules provide targeted benefits in sparse or high-risk slices, but the full disruption gate is not the overall winner."
)
report_lines.append("")
report_lines.append("## Table 1: Dataset summary")
report_lines.append(dataframe_to_text_table(dataset_summary, ["item", "value"], max_rows=80))
report_lines.append("")
report_lines.append("## Table 2: Temporal baselines")
report_lines.append(dataframe_to_text_table(table_temporal, ["paper_table_group", "paper_model", "pinball_mean", "mae_q75", "iqr_mae", "experiment_id"], max_rows=20))
report_lines.append("")
report_lines.append("## Table 3: Cold-OD main results")
report_lines.append(dataframe_to_text_table(table_cold_od, ["paper_table_group", "paper_model", "pinball_mean", "mae_q75", "iqr_mae", "stress_top10_mae_q75"], max_rows=20))
report_lines.append("")
report_lines.append("## Table 4: Graph ablation")
report_lines.append(dataframe_to_text_table(table_graph_ablation, ["paper_table_group", "model", "checkpoint_metric", "seed", "pinball_mean", "mae_q75", "iqr_mae"], max_rows=30))
report_lines.append("")
report_lines.append("## Table 5: Topology ablation")
report_lines.append(dataframe_to_text_table(table_topology, ["paper_table_group", "model", "checkpoint_metric", "seed", "pinball_mean", "mae_q75", "iqr_mae"], max_rows=30))
report_lines.append("")
report_lines.append("## Table 6: Disruption ablation")
report_lines.append(dataframe_to_text_table(table_disruption, ["paper_table_group", "model", "checkpoint_metric", "seed", "pinball_mean", "mae_q75", "iqr_mae"], max_rows=30))
report_lines.append("")
report_lines.append("## Table 7: Calibration")
report_lines.append(dataframe_to_text_table(table_calibration, ["paper_model", "source", "model", "calibration_method", "pinball_mean", "mae_q75", "iqr_mae"], max_rows=20))
report_lines.append("")
report_lines.append("## Table 8: Decision-aware CVaR screening")
report_lines.append(dataframe_to_text_table(table_cvar, list(table_cvar.columns), max_rows=20))
report_lines.append("")
report_lines.append("## KDD-oriented narrative")
report_lines.append("")
report_lines.append(
    "FREIGHT-MNet should be framed as a public multimodal freight graph benchmark for OD-level distributional truck travel-time risk prediction. "
    "The main story is not that one full model wins every metric. Instead, historical priors dominate recurring OD relations, graph learning dominates unseen OD relations, "
    "topology helps selected sparse/high-risk slices, weather disruption helps selected high-q75 tail errors, and decision-aware risk screening favors GraphSAGE."
)

report_path = paths.reports_dir / "final_results_consolidation_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved final report:", report_path)
print("\n".join(report_lines[:40]))

Saved final report: E:\NetworkOptimization\pythonProject1\Data\10_experiments\final_results_consolidation_v1_notebook\east_plus_gulf\reports\final_results_consolidation_report.md
# FREIGHT-MNet Final Results Consolidation Report

Created at UNIX time: 1779411609.965
Data root: E:\NetworkOptimization\pythonProject1\Data
Scope: east_plus_gulf

## Executive interpretation

The core experimental chain is complete. The strongest empirical result is Cold-OD graph generalization: GraphV2 HGT is the best overall pinball model, while GraphV2 GraphSAGE is the strongest q75/IQR and decision-aware risk-screening model. Topology and weather-disruption modules provide targeted benefits in sparse or high-risk slices, but the full disruption gate is not the overall winner.

## Table 1: Dataset summary
                          item                           value
               supervised_rows                           73972
                unique_origins                             104
           uni

## 13. Save run configuration and artifact manifest

In [16]:
artifact_manifest = {
    "output_dir": str(paths.output_dir),
    "tables_dir": str(paths.tables_dir),
    "plots_dir": str(paths.plots_dir),
    "reports_dir": str(paths.reports_dir),
    "artifact_inventory": str(paths.tables_dir / "artifact_inventory.csv"),
    "all_loaded_metrics_normalized": str(paths.tables_dir / "all_loaded_metrics_normalized.csv"),
    "table_1_dataset_summary": str(paths.tables_dir / "table_1_dataset_summary.csv"),
    "table_2_temporal_baselines": str(paths.tables_dir / "table_2_temporal_baselines.csv"),
    "table_3_cold_od_main_results": str(paths.tables_dir / "table_3_cold_od_main_results.csv"),
    "table_4_graph_ablation": str(paths.tables_dir / "table_4_graph_ablation.csv"),
    "table_5_topology_ablation": str(paths.tables_dir / "table_5_topology_ablation.csv"),
    "table_6_disruption_ablation": str(paths.tables_dir / "table_6_disruption_ablation.csv"),
    "table_7_calibration_summary": str(paths.tables_dir / "table_7_calibration_summary.csv"),
    "table_8_decision_cvar_screening": str(paths.tables_dir / "table_8_decision_cvar_screening.csv"),
    "normalized_final_predictions": str(paths.output_dir / "normalized_final_predictions_val_test.parquet"),
    "final_sparse_segment_audit": str(paths.tables_dir / "final_sparse_segment_audit.csv"),
    "final_report": str(paths.reports_dir / "final_results_consolidation_report.md"),
}

run_config = {
    "notebook": "Final_Results_Consolidation",
    "created_at_unix": now_unix(),
    "config": asdict(cfg),
    "paths": {
        "data_root": str(cfg.data_root),
        "supervised_model_ready_path": str(paths.supervised_model_ready_path),
        "supervised_with_disruption_path": str(paths.supervised_with_disruption_path),
        "manifest_path": str(paths.manifest_path),
        "full_heterodata_path": str(paths.full_heterodata_path),
        "topology_features_path": str(paths.topology_features_path),
        "output_dir": str(paths.output_dir),
    },
    "package_versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib_available": MATPLOTLIB_AVAILABLE,
    },
}

write_json(artifact_manifest, paths.output_dir / "analysis_artifact_manifest_final_results_consolidation.json")
write_json(run_config, paths.output_dir / "run_config_final_results_consolidation.json")

print(json.dumps(artifact_manifest, indent=2))

{
  "output_dir": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\final_results_consolidation_v1_notebook\\east_plus_gulf",
  "tables_dir": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\final_results_consolidation_v1_notebook\\east_plus_gulf\\tables",
  "plots_dir": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\final_results_consolidation_v1_notebook\\east_plus_gulf\\plots",
  "reports_dir": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\final_results_consolidation_v1_notebook\\east_plus_gulf\\reports",
  "artifact_inventory": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\final_results_consolidation_v1_notebook\\east_plus_gulf\\tables\\artifact_inventory.csv",
  "all_loaded_metrics_normalized": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\final_results_consolidation_v1_notebook\\east_plus_gulf\\tables\\all_loaded_metrics_normalized.csv",
  "table_1_dataset_summary": "E:\\NetworkOpt